# Colab bootstrap

Run the cell below once after connecting to a new Colab runtime. It mounts Google Drive, clones or updates the repository, installs the project with the annotation-app dependencies, and configures the shared runtime paths.

In [2]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/terriljoel/retrieval-grounded-remote-sensing.git"
REPO_REF = "feat/object-detection-YOLO"  # Change to main after merging.
REPO_DIR = Path("/content/retrieval-grounded-remote-sensing")
SHARED_ROOT = Path("/content/drive/Othercomputers/My laptop/shared_resources")

drive.mount("/content/drive")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Repository path exists but is not a Git clone: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[annotation]"],
    check=True,
)

if not SHARED_ROOT.is_dir():
    raise FileNotFoundError(
        f"Shared resources were not found at {SHARED_ROOT}. "
        "Check the Google Drive computer and folder names."
    )

os.environ.update({
    "SHARED_RESOURCES_ROOT": str(SHARED_ROOT),
    "RAW_DATASET_ROOT": "/content/datasets/raw",
    "PROCESSED_DATASET_ROOT": "/content/datasets/processed",
    "MANIFEST_ROOT": str(SHARED_ROOT / "datasets" / "manifests"),
    "EXPERIMENT_OUTPUT_ROOT": str(SHARED_ROOT / "experiment_outputs"),
    "JOB_LOG_ROOT": str(SHARED_ROOT / "experiment_outputs" / "job_logs"),
    "INFERENCE_EXPORT_ROOT": str(SHARED_ROOT / "inference" / "object_detection_inference_new_remote_sensing_dataset_external-4"),
})

Path(os.environ["MANIFEST_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["EXPERIMENT_OUTPUT_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["JOB_LOG_ROOT"]).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

print(f"Ready. Working directory: {Path.cwd()}")
print("The detector commands and annotation app are ready.")

Mounted at /content/drive
Ready. Working directory: /content/retrieval-grounded-remote-sensing
The detector commands and annotation app are ready.


# Object Detection Study

## Prepare Dataset


NWPU ZIP/raw files
        
Locate or extract dataset
        
Reuse/perform audit
        
Reuse/create train–validation–test split
        
Convert NWPU boxes to YOLO labels
        
Validate converted dataset
        
Write dataset.yaml

In [3]:
!rs-prepare-detector --config configs/detection/yolov8n_scratch_640.yaml

[job] id=20260920T144643Z_a3694458
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-prepare-detector/20260920T144643Z_a3694458
Reused dataset audit: /content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/data_preparation/nwpu_audit.json
Reused split manifest: /content/drive/Othercomputers/My laptop/shared_resources/datasets/manifests/nwpu_vhr10_multilabel_split_seed42.csv
Prepared YOLO dataset: /content/datasets/processed/nwpu_vhr10_yolo_seed42
{
  "images": 800,
  "labels": 800,
  "objects": 3896,
  "backgrounds": 150,
  "class_counts": {
    "0": 757,
    "1": 302,
    "2": 655,
    "3": 390,
    "4": 524,
    "5": 159,
    "6": 163,
    "7": 224,
    "8": 124,
    "9": 598
  },
  "issues": [],
  "valid": true
}
[job] status=succeeded log=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-prepare-detector/20260920T144643Z_a3694458/run.log


## E1 Training Model From Scratch

In [5]:
!rs-train-detector --config configs/detection/yolov8n_scratch_640.yaml

[job] id=20260919T194428Z_b3d88168
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-train-detector/20260919T194428Z_b3d88168
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/processed/nwpu_vhr10_yolo_seed42/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl

### Evaluate the model

In [6]:
%cd /content/retrieval-grounded-remote-sensing

CONFIG = "configs/detection/yolov8n_scratch_640.yaml"
CHECKPOINT = (
    "/content/drive/Othercomputers/My laptop/shared_resources/"
    "experiment_outputs/detection/"
    "yolov8n_scratch_640_seed42/weights/best.pt"
)

!rs-evaluate-detector --config "$CONFIG" --checkpoint "$CHECKPOINT" --confirm-test

/content
[job] id=20260919T195931Z_e8c7fea6
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-evaluate-detector/20260919T195931Z_e8c7fea6
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1607.2±566.1 MB/s, size: 91.9 KB)
val: Scanning /content/datasets/processed/nwpu_vhr10_yolo_seed42/labels/test... 118 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 1.6Kit/s 0.1s
val: New cache created: /content/datasets/processed/nwpu_vhr10_yolo_seed42/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.3it/s 3.5s0.2ss
                   all        118        523      0.778      0.574      0.695      0.363
              airplane         13        101      0.897      0.861      0.942      0.49

## E2 Training Model with higher resolution images

In [7]:
CONFIG = "configs/detection/yolov8n_pretrained_1024.yaml"
CHECKPOINT = (
    "/content/drive/Othercomputers/My laptop/shared_resources/"
    "experiment_outputs/detection/"
    "yolov8n_pretrained_1024_seed42/weights/best.pt"
)

In [8]:
!rs-train-detector --config "$CONFIG"

[job] id=20260919T200740Z_a4002da1
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-train-detector/20260919T200740Z_a4002da1
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/processed/nwpu_vhr10_yolo_seed42/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=Fals

### Evaluate the model

In [9]:
# %cd /content/retrieval-grounded-remote-sensing

!rs-evaluate-detector --config "$CONFIG" --checkpoint "$CHECKPOINT" --confirm-test

[job] id=20260919T202650Z_cb183574
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-evaluate-detector/20260919T202650Z_cb183574
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2270.9±804.5 MB/s, size: 92.4 KB)
val: Scanning /content/datasets/processed/nwpu_vhr10_yolo_seed42/labels/test.cache... 118 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 13.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 5.3it/s 2.8s0.1s
                   all        118        523      0.945      0.929      0.957      0.653
              airplane         13        101      0.974       0.99      0.995      0.697
                  ship          9         48      0.924      0.812      0.842      0.569
     

## E3a fine tuning Model with more epochs

In [10]:
CONFIG = "configs/detection/yolov8n_pretrained_640_100.yaml"
CHECKPOINT = (
    "/content/drive/Othercomputers/My laptop/shared_resources/"
    "experiment_outputs/detection/"
    "yolov8n_pretrained_640_100_seed42/weights/best.pt"
)

In [11]:
!rs-train-detector --config "$CONFIG"

[job] id=20260919T202705Z_649ed5e8
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-train-detector/20260919T202705Z_649ed5e8
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/processed/nwpu_vhr10_yolo_seed42/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=Fal

### Evaluate the model

In [12]:
# %cd /content/retrieval-grounded-remote-sensing

!rs-evaluate-detector --config "$CONFIG" --checkpoint "$CHECKPOINT" --confirm-test

[job] id=20260919T204605Z_15e75255
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-evaluate-detector/20260919T204605Z_15e75255
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2621.8±929.9 MB/s, size: 80.5 KB)
val: Scanning /content/datasets/processed/nwpu_vhr10_yolo_seed42/labels/test.cache... 118 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 22.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.9it/s 4.2s0.4ss
                   all        118        523      0.913      0.935      0.944      0.638
              airplane         13        101      0.967          1      0.994      0.697
                  ship          9         48      0.975      0.824       0.89      0.566
      

## E3b fine tuning Model with more epochs and Cosine LR scheduler

In [13]:
CONFIG = "configs/detection/yolov8n_pretrained_640_100_cosine.yaml"
CHECKPOINT = (
    "/content/drive/Othercomputers/My laptop/shared_resources/"
    "experiment_outputs/detection/"
    "yolov8n_pretrained_640_100_cosine_seed42/weights/best.pt"
)

In [14]:
!rs-train-detector --config "$CONFIG"

[job] id=20260919T204620Z_11dbf864
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-train-detector/20260919T204620Z_11dbf864
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/datasets/processed/nwpu_vhr10_yolo_seed42/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=Fals

### Evaluate the model

In [15]:
# %cd /content/retrieval-grounded-remote-sensing

!rs-evaluate-detector --config "$CONFIG" --checkpoint "$CHECKPOINT" --confirm-test

[job] id=20260919T205623Z_3d6615da
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-evaluate-detector/20260919T205623Z_3d6615da
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2749.9±386.3 MB/s, size: 102.8 KB)
val: Scanning /content/datasets/processed/nwpu_vhr10_yolo_seed42/labels/test.cache... 118 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 22.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.1it/s 2.6s0.2s
                   all        118        523       0.92      0.884      0.931      0.605
              airplane         13        101      0.981       0.98      0.995      0.675
                  ship          9         48      0.964      0.812      0.863      0.526
      

## E3c Training from scratch with more epochs

In [20]:
CONFIG = "configs/detection/yolov8n_scratch_640_100.yaml"
CHECKPOINT = (
    "/content/drive/Othercomputers/My laptop/shared_resources/"
    "experiment_outputs/detection/"
    "yolov8n_scratch_640_100_seed42/weights/best.pt"
)

In [21]:
!rs-train-detector --config "$CONFIG"

[job] id=20260919T211039Z_a80452a7
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-train-detector/20260919T211039Z_a80452a7
New https://pypi.org/project/ultralytics/8.4.156 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/processed/nwpu_vhr10_yolo_seed42/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=to

: 

### Evaluate the model

In [ ]:
# %cd /content/retrieval-grounded-remote-sensing

!rs-evaluate-detector --config "$CONFIG" --checkpoint "$CHECKPOINT" --confirm-test

[job] id=20260919T205623Z_3d6615da
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-evaluate-detector/20260919T205623Z_3d6615da
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2749.9±386.3 MB/s, size: 102.8 KB)
val: Scanning /content/datasets/processed/nwpu_vhr10_yolo_seed42/labels/test.cache... 118 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 22.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.1it/s 2.6s0.2s
                   all        118        523       0.92      0.884      0.931      0.605
              airplane         13        101      0.981       0.98      0.995      0.675
                  ship          9         48      0.964      0.812      0.863      0.526
      

## E4 Different Model

In [16]:
CONFIG = "configs/detection/yolov8s_pretrained_640.yaml"
CHECKPOINT = (
    "/content/drive/Othercomputers/My laptop/shared_resources/"
    "experiment_outputs/detection/"
    "yolov8s_pretrained_640_seed42/weights/best.pt"
)

In [17]:
!rs-train-detector --config "$CONFIG"

[job] id=20260919T205638Z_e2cff479
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-train-detector/20260919T205638Z_e2cff479
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/processed/nwpu_vhr10_yolo_seed42/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False

### Evaluate the model

In [18]:
# %cd /content/retrieval-grounded-remote-sensing

!rs-evaluate-detector --config "$CONFIG" --checkpoint "$CHECKPOINT" --confirm-test

[job] id=20260919T210802Z_36893dd4
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-evaluate-detector/20260919T210802Z_36893dd4
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2773.1±565.9 MB/s, size: 96.0 KB)
val: Scanning /content/datasets/processed/nwpu_vhr10_yolo_seed42/labels/test.cache... 118 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 18.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 4.0it/s 3.7s0.2s
                   all        118        523      0.949      0.955      0.972      0.643
              airplane         13        101      0.969       0.99      0.995      0.693
                  ship          9         48      0.967      0.854      0.863      0.557
   

In [2]:
!ls

datasets  drive  retrieval-grounded-remote-sensing  sample_data


In [1]:
!git pull

fatal: not a git repository (or any of the parent directories): .git


## E5 Yolov8s 1024 pixel input size

In [3]:
CONFIG = "configs/detection/yolov8s_pretrained_1024.yaml"
CHECKPOINT = (
    "/content/drive/Othercomputers/My laptop/shared_resources/"
    "experiment_outputs/detection/"
    "yolov8s_pretrained_1024_seed42/weights/best.pt"
)

In [7]:
%cd /content/retrieval-grounded-remote-sensing

/content/retrieval-grounded-remote-sensing


In [11]:
!rs-train-detector --config "$CONFIG"

[job] id=20260919T213326Z_7dd2cf35
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-train-detector/20260919T213326Z_7dd2cf35
New https://pypi.org/project/ultralytics/8.4.156 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/processed/nwpu_vhr10_yolo_seed42/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=tor

### Evaluate the model

In [12]:
# %cd /content/retrieval-grounded-remote-sensing

!rs-evaluate-detector --config "$CONFIG" --checkpoint "$CHECKPOINT" --confirm-test

[job] id=20260919T215814Z_b9e6e9c5
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-evaluate-detector/20260919T215814Z_b9e6e9c5
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1987.9±1023.2 MB/s, size: 83.2 KB)
val: Scanning /content/datasets/processed/nwpu_vhr10_yolo_seed42/labels/test.cache... 118 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 21.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 30/30 8.5it/s 3.5s0.1ss
                   all        118        523      0.932       0.92      0.956      0.659
              airplane         13        101      0.978       0.98      0.994      0.703
                  ship          9         48      0.861      0.833      0.905       0.59
 

## E6 Yolov8s 1024 pixel input size and batch size 16

In [4]:
CONFIG = "configs/detection/yolov8s_pretrained_1024_16.yaml"
CHECKPOINT = (
    "/content/drive/Othercomputers/My laptop/shared_resources/"
    "experiment_outputs/detection/"
    "yolov8s_pretrained_1024_seed42/weights/best.pt"
)

In [ ]:
%cd /content/retrieval-grounded-remote-sensing

/content/retrieval-grounded-remote-sensing


In [5]:
!rs-train-detector --config "$CONFIG"

[job] id=20260920T144738Z_2a1a6ecb
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-train-detector/20260920T144738Z_2a1a6ecb
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/processed/nwpu_vhr10_yolo_seed42/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl

### Evaluate the model

In [6]:
# %cd /content/retrieval-grounded-remote-sensing

!rs-evaluate-detector --config "$CONFIG" --checkpoint "$CHECKPOINT" --confirm-test

[job] id=20260920T151320Z_2595ca5c
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-evaluate-detector/20260920T151320Z_2595ca5c
Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 22.2±9.8 MB/s, size: 90.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/datasets/processed/nwpu_vhr10_yolo_seed42/labels/test... 118 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 118/118 510.5it/s 0.2s0.1s
val: New cache created: /content/datasets/processed/nwpu_vhr10_yolo_seed42/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.3it/s 6.1s0.6ss
                   all       